# Limit-order strategy research

This notebook records candidate approaches and the data contract needed to test them honestly. The current Shiller monthly engine cannot establish whether an intramonth limit price was touched, in what sequence prices traded, or whether a gap produced a better fill. No fill-performance claims should be made from monthly averages.


In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from retail_sp500.strategies import strategy_catalog


## Candidate limit-setting approaches

All approaches remain long-only: unfilled cash stays in cash, and filled orders buy the S&P 500 ETF.


In [ ]:
approaches = pd.DataFrame([
    {
        "approach": "Fixed discount",
        "example": "Buy at 0.5%, 1%, or 2% below the previous close",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Misses persistent rallies",
    },
    {
        "approach": "Limit ladder",
        "example": "Split cash across -0.5%, -1%, -2%, and -4% orders",
        "required_data": "Adjusted daily OHLC and order state",
        "main_risk": "Deep orders rarely fill",
    },
    {
        "approach": "Volatility-scaled discount",
        "example": "Set distance as a multiple of ATR or recent realized volatility",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Limits become too distant during shocks",
    },
    {
        "approach": "Drawdown ladder",
        "example": "Deploy tranches at 5%, 10%, 15%, and 20% below the recent peak",
        "required_data": "Adjusted daily close/high and persistent cash state",
        "main_risk": "Cash drag in long bull markets",
    },
    {
        "approach": "Moving-average pullback",
        "example": "Place limits near the 20-day, 50-day, or 200-day average",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Averages lag structural breaks",
    },
    {
        "approach": "Bollinger or z-score",
        "example": "Buy at a lower rolling standard-deviation band",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Repeated fills in a falling market",
    },
    {
        "approach": "RSI-gated limit",
        "example": "Activate a discount order only after an oversold RSI reading",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Parameter sensitivity and short sample edge",
    },
    {
        "approach": "Trend-gated limit",
        "example": "Use pullback limits only while the long-term trend is positive",
        "required_data": "Adjusted daily OHLC",
        "main_risk": "Misses early rebounds after trend exits",
    },
    {
        "approach": "Time-decaying limit",
        "example": "Begin below market, then move the limit upward until expiry",
        "required_data": "Daily OHLC and explicit order lifecycle",
        "main_risk": "Converges toward market order after delays",
    },
    {
        "approach": "Cash-age urgency",
        "example": "Use deeper limits for fresh cash and tighten them as cash ages",
        "required_data": "Daily OHLC and contribution-lot tracking",
        "main_risk": "More execution-state complexity",
    },
])

approaches


## Minimum execution contract

A defensible daily limit-order engine should include:

- split-adjusted open, high, low and close;
- separate dividends or a consistent total-return accounting path;
- order placement time, expiry and cancellation rules;
- gap handling: a buy limit above the open fills at the open, not automatically at the limit;
- conservative ambiguity handling when both entry and exit levels are touched in one bar;
- transaction costs, ETF expense ratio and cash yield;
- comparison against investing the same cash immediately;
- no use of future high/low information when setting the order.


In [ ]:
current_keys = list(strategy_catalog())
assert not any("limit" in key for key in current_keys)

current_keys


## Recommended implementation order

1. Add a separately validated adjusted-daily OHLC loader.
2. Add an execution engine with persistent cash, units, pending orders and conservative fill rules.
3. Start with fixed-discount and limit-ladder baselines.
4. Add volatility-scaled and trend-gated variants.
5. Evaluate rolling periods, fill rate, cash drag, turnover and terminal wealth against immediate monthly investment.

Do not mix these results into the 1871-present monthly graph until both engines expose their different data horizons clearly.
